# Wind Generation Reliability Analysis

This notebook examines the **reliability and variability** of GB wind generation during **January 2024**, using half-hourly outturn data from the Elexon BMRS API.

We answer key questions for system operators and traders:
- How much wind generation is reliably available?
- What does the generation duration curve look like?
- What capacity factor does wind achieve?
- How variable is output (ramp rates, hourly patterns)?

**Data source:** Elexon BMRS `FUELHH` dataset (fuelType=WIND)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import warnings

warnings.filterwarnings("ignore")

# Professional styling
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# Assumed installed wind capacity (onshore + offshore) for GB as of Jan 2024
INSTALLED_CAPACITY_MW = 30_000

print("Libraries loaded successfully.")
print(f"Assumed installed wind capacity: {INSTALLED_CAPACITY_MW:,} MW")

## 2. Fetch Wind Generation Data

In [ ]:
URL = (
    "https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH/stream"
    "?fuelType=WIND&settlementDateFrom=2024-01-01&settlementDateTo=2024-01-31"
)

print("Fetching wind generation data from Elexon FUELHH ...")
response = requests.get(URL, timeout=120)
response.raise_for_status()

df = pd.DataFrame(response.json())
print(f"Rows received: {len(df):,}")
df.head()

In [ ]:
# Normalise column names
df.columns = df.columns.str.strip()
print("Columns:", df.columns.tolist())

# Detect key columns
time_col = [c for c in df.columns if "start" in c.lower() and "time" in c.lower()]
gen_col = [c for c in df.columns if "generation" in c.lower() or "quantity" in c.lower()]

if time_col and gen_col:
    df = df.rename(columns={time_col[0]: "startTime", gen_col[0]: "generation_mw"})
else:
    print("Warning: Could not auto-detect columns. Printing sample for inspection.")
    print(df.head())

df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
df["generation_mw"] = pd.to_numeric(df["generation_mw"], errors="coerce")
df = df.dropna(subset=["generation_mw"]).sort_values("startTime").reset_index(drop=True)

print(f"\nTime range: {df['startTime'].min()} to {df['startTime'].max()}")
print(f"Records: {len(df):,}")
print(f"\nGeneration statistics (MW):")
df["generation_mw"].describe().round(0)

## 3. Generation Duration Curve

The duration curve sorts generation values in descending order, showing what fraction of time output exceeds a given level.  This is a cornerstone of power system adequacy analysis.

In [ ]:
sorted_gen = df["generation_mw"].sort_values(ascending=False).reset_index(drop=True)
n = len(sorted_gen)
pct_time = np.arange(1, n + 1) / n * 100  # % of time exceeded

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(pct_time, sorted_gen, alpha=0.3, color="steelblue")
ax.plot(pct_time, sorted_gen, linewidth=2, color="steelblue")

# Mark key percentiles
for p, label, color in [(10, "p10 (exceeded 90%)", "tab:red"),
                         (50, "p50 (median)", "tab:orange"),
                         (90, "p90 (exceeded 10%)", "tab:green")]:
    val = np.percentile(sorted_gen, 100 - p)
    ax.axhline(val, linestyle="--", color=color, alpha=0.7, linewidth=1.2)
    ax.axvline(p, linestyle=":", color=color, alpha=0.4)
    ax.annotate(f"{label}: {val:,.0f} MW",
                xy=(p, val), xytext=(p + 3, val + 500),
                fontsize=9, fontweight="bold", color=color,
                arrowprops=dict(arrowstyle="->", color=color, lw=1))

ax.set_xlabel("% of Time Exceeded")
ax.set_ylabel("Wind Generation (MW)")
ax.set_title("Wind Generation Duration Curve - January 2024")
ax.set_xlim(0, 100)
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
fig.tight_layout()
plt.show()

## 4. Capacity Factor Analysis

The **capacity factor** is the ratio of actual generation to the maximum possible output if all installed capacity ran at full power.  We assume approximately **30 GW** of installed wind capacity across GB.

In [ ]:
mean_gen = df["generation_mw"].mean()
capacity_factor = mean_gen / INSTALLED_CAPACITY_MW * 100

print(f"Mean generation:       {mean_gen:>10,.0f} MW")
print(f"Installed capacity:    {INSTALLED_CAPACITY_MW:>10,} MW")
print(f"Capacity factor:       {capacity_factor:>9.1f} %")
print(f"Peak generation:       {df['generation_mw'].max():>10,.0f} MW")
print(f"Peak utilisation:      {df['generation_mw'].max() / INSTALLED_CAPACITY_MW * 100:>9.1f} %")
print(f"Minimum generation:    {df['generation_mw'].min():>10,.0f} MW")

In [ ]:
# Daily capacity factor
df["date"] = df["startTime"].dt.date
daily = df.groupby("date")["generation_mw"].mean().reset_index()
daily["capacity_factor_pct"] = daily["generation_mw"] / INSTALLED_CAPACITY_MW * 100

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(daily)), daily["capacity_factor_pct"],
              color=sns.color_palette("coolwarm_r", len(daily)), edgecolor="white", linewidth=0.5)
ax.axhline(capacity_factor, color="black", linestyle="--", linewidth=1.2, label=f"Monthly avg: {capacity_factor:.1f}%")
ax.set_xticks(range(0, len(daily), 2))
ax.set_xticklabels([str(d) for d in daily["date"].iloc[::2]], rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Date")
ax.set_ylabel("Capacity Factor (%)")
ax.set_title("Daily Wind Capacity Factor - January 2024")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Percentile Analysis

Percentiles tell us the MW level exceeded a given fraction of the time, which is critical for security-of-supply assessments.

In [ ]:
percentiles = [5, 10, 25, 50, 75, 90, 95]
pct_values = df["generation_mw"].quantile([p / 100 for p in percentiles])
pct_df = pd.DataFrame({
    "Percentile": [f"p{p}" for p in percentiles],
    "Generation (MW)": pct_values.values.round(0).astype(int),
    "% Time Exceeded": [f"{100 - p}%" for p in percentiles],
    "Capacity Factor (%)": (pct_values.values / INSTALLED_CAPACITY_MW * 100).round(1),
})

print("Percentile Analysis - Wind Generation January 2024")
print("=" * 60)
pct_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = sns.color_palette("viridis", len(percentiles))
bars = ax.barh([f"p{p}" for p in percentiles], pct_values.values, color=colors, edgecolor="black", height=0.6)

for bar, val in zip(bars, pct_values.values):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f} MW", va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Wind Generation (MW)")
ax.set_title("Wind Generation Percentiles - January 2024")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.set_xlim(right=pct_values.max() * 1.2)
fig.tight_layout()
plt.show()

## 6. Variability Analysis

### 6.1 Ramp Rates

Ramp rates measure the 30-minute change in generation (MW).  Large ramps stress the system and require flexible backup.

In [ ]:
df["ramp_mw"] = df["generation_mw"].diff()
df["abs_ramp"] = df["ramp_mw"].abs()

ramp_stats = df["ramp_mw"].dropna().describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

print("30-Minute Ramp Rate Statistics (MW)")
print("=" * 40)
print(ramp_stats.round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of ramp rates
ax = axes[0]
ax.hist(df["ramp_mw"].dropna(), bins=80, color="steelblue", edgecolor="white", linewidth=0.3, alpha=0.8)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("30-min Ramp Rate (MW)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of 30-Minute Wind Ramp Rates")

# Time series of ramp rates
ax = axes[1]
ax.plot(df["startTime"], df["ramp_mw"], linewidth=0.5, alpha=0.7, color="steelblue")
ax.axhline(0, color="black", linewidth=0.8)
p99_up = df["ramp_mw"].quantile(0.99)
p01_down = df["ramp_mw"].quantile(0.01)
ax.axhline(p99_up, color="tab:red", linestyle="--", label=f"p99 up: {p99_up:+,.0f} MW")
ax.axhline(p01_down, color="tab:red", linestyle="--", label=f"p1 down: {p01_down:+,.0f} MW")
ax.set_xlabel("Date")
ax.set_ylabel("Ramp Rate (MW per 30 min)")
ax.set_title("Wind Ramp Rates Over Time")
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

### 6.2 Variability by Hour of Day

Standard deviation of generation by hour of day reveals when output is most and least predictable.

In [ ]:
df["hour"] = df["startTime"].dt.hour

hourly_stats = df.groupby("hour")["generation_mw"].agg(["mean", "std", "min", "max"]).reset_index()
hourly_stats.columns = ["Hour", "Mean (MW)", "Std Dev (MW)", "Min (MW)", "Max (MW)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean generation with std dev band
ax = axes[0]
ax.plot(hourly_stats["Hour"], hourly_stats["Mean (MW)"], marker="o", linewidth=2, color="steelblue")
ax.fill_between(hourly_stats["Hour"],
                hourly_stats["Mean (MW)"] - hourly_stats["Std Dev (MW)"],
                hourly_stats["Mean (MW)"] + hourly_stats["Std Dev (MW)"],
                alpha=0.2, color="steelblue", label="+/- 1 std dev")
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Generation (MW)")
ax.set_title("Mean Wind Generation by Hour")
ax.legend()
ax.set_xticks(range(0, 24, 2))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Std dev by hour
ax = axes[1]
ax.bar(hourly_stats["Hour"], hourly_stats["Std Dev (MW)"], color="coral", edgecolor="black", width=0.7)
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Std Deviation (MW)")
ax.set_title("Wind Generation Variability by Hour")
ax.set_xticks(range(0, 24, 2))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.tight_layout()
plt.show()

hourly_stats.round(0)

## 7. Reliable Wind Contribution (p10 Recommendation)

For capacity adequacy and firm supply planning, the **p10 level** (exceeded 90% of the time) is a common benchmark for "reliable" wind contribution.

This MW level can be counted on with high confidence across the month.

In [ ]:
p10_value = df["generation_mw"].quantile(0.10)
p5_value = df["generation_mw"].quantile(0.05)
p25_value = df["generation_mw"].quantile(0.25)

print("=" * 60)
print("  RELIABLE WIND CONTRIBUTION RECOMMENDATION")
print("  January 2024")
print("=" * 60)
print(f"")
print(f"  p5  (exceeded 95% of time):  {p5_value:>8,.0f} MW  ({p5_value/INSTALLED_CAPACITY_MW*100:.1f}% CF)")
print(f"  p10 (exceeded 90% of time):  {p10_value:>8,.0f} MW  ({p10_value/INSTALLED_CAPACITY_MW*100:.1f}% CF)  <-- RECOMMENDED")
print(f"  p25 (exceeded 75% of time):  {p25_value:>8,.0f} MW  ({p25_value/INSTALLED_CAPACITY_MW*100:.1f}% CF)")
print(f"")
print(f"  Recommendation: For capacity planning in winter months,")
print(f"  count on approximately {p10_value:,.0f} MW of wind generation")
print(f"  as a firm contribution (p10 = available 90% of the time).")
print(f"")
print(f"  This represents {p10_value/INSTALLED_CAPACITY_MW*100:.1f}% of the ~{INSTALLED_CAPACITY_MW/1000:.0f} GW installed capacity.")
print(f"  The remaining {INSTALLED_CAPACITY_MW - p10_value:,.0f} MW of nameplate capacity should")
print(f"  be backed by dispatchable reserves or demand-side response.")
print("=" * 60)

In [ ]:
# Final visualisation: generation time series with reliability bands
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df["startTime"], df["generation_mw"], linewidth=0.6, alpha=0.8, color="steelblue", label="Actual generation")

ax.axhline(p10_value, color="tab:red", linestyle="--", linewidth=1.5,
           label=f"p10 = {p10_value:,.0f} MW (reliable floor)")
ax.axhline(mean_gen, color="tab:orange", linestyle="-.", linewidth=1.5,
           label=f"Mean = {mean_gen:,.0f} MW")
ax.axhline(INSTALLED_CAPACITY_MW, color="grey", linestyle=":", linewidth=1,
           label=f"Installed capacity = {INSTALLED_CAPACITY_MW/1000:.0f} GW")

ax.fill_between(df["startTime"], 0, p10_value, alpha=0.1, color="tab:red")

ax.set_xlabel("Date")
ax.set_ylabel("Wind Generation (MW)")
ax.set_title("Wind Generation with Reliability Floor (p10) - January 2024")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(bottom=0)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.tight_layout()
plt.show()

## 8. Summary

Key findings from the January 2024 wind reliability analysis:

1. **Duration curve**: Wind output varies enormously; the generation duration curve falls steeply, meaning high output is only sustained for a small fraction of time.
2. **Capacity factor**: The monthly capacity factor reflects typical winter wind conditions.
3. **Percentiles**: The p10 level provides a defensible estimate of firm wind contribution for system adequacy.
4. **Ramp rates**: 30-minute ramps can be substantial, requiring significant flexible backup.
5. **Hourly patterns**: Wind variability (std deviation) is relatively stable across hours, unlike solar which has strong diurnal patterns.
6. **Recommendation**: For capacity planning, only the p10 level should be treated as firm -- the rest must be backed by dispatchable generation, storage, or demand response.